# Huấn luyện AI nhận diện vật thể cho ScreenWatcher (bản 2)
Colab mới dùng Python 3.13 nên không cài trực tiếp được MediaPipe Model Maker (cần Python 3.9–3.12 và TensorFlow 2.15).
Bản này tự tạo môi trường **Python 3.11** riêng rồi chạy huấn luyện trong đó.
Chạy lần lượt từng ô (chạm vào `[ ]` bên trái ô để chạy). Bật GPU: **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU**.
Cần file `dataset_coco.zip` xuất từ app (🏷 Gán nhãn ảnh → 📤 Xuất COCO).

## 1. Tạo môi trường Python 3.11 và cài thư viện (vài phút)

In [ ]:
!pip -q install uv
!rm -rf /content/venv
!uv venv --python 3.11 /content/venv
# cài MỘT lượt, ghim numpy<2 ngay từ đầu để các thư viện (ml_dtypes, opencv...) chọn bản tương thích với TensorFlow 2.15
!uv pip install --python /content/venv/bin/python "numpy<2" "tensorflow[and-cuda]==2.15.1" "mediapipe==0.10.14" mediapipe-model-maker "setuptools<81"
!MPLBACKEND=Agg /content/venv/bin/python -c "from mediapipe_model_maker import object_detector; print('Model Maker OK')"
!MPLBACKEND=Agg /content/venv/bin/python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))"


## 2. Tải bộ dữ liệu lên
Bấm *Choose files* và chọn `dataset_coco.zip`.

In [ ]:
from google.colab import files
up = files.upload()
!rm -rf dataset && mkdir dataset
!unzip -q -o dataset_coco.zip -d dataset
# zip xuất bởi bản app cũ dùng thư mục "data"; Model Maker cần "images"
!for s in train validation; do if [ -d dataset/$s/data ]; then mv dataset/$s/data dataset/$s/images; fi; done
!ls dataset dataset/train | head

## 3. Viết kịch bản huấn luyện (chỉ chạy ô, không cần sửa)

In [ ]:
%%writefile train.py
import os, json
os.environ["MPLBACKEND"] = "Agg"  # Colab đặt backend inline, môi trường 3.11 không có -> báo lỗi
import tensorflow as tf
from mediapipe_model_maker import object_detector

EPOCHS = 50   # tăng lên 80-100 nếu kết quả chưa tốt

train_data = object_detector.Dataset.from_coco_folder("dataset/train", cache_dir="/tmp/od_data/train")
validation_data = object_detector.Dataset.from_coco_folder("dataset/validation", cache_dir="/tmp/od_data/validation")
print("Ảnh huấn luyện:", train_data.size, "| ảnh kiểm tra:", validation_data.size)
print("Các loại:", [c["name"] for c in json.load(open("dataset/train/labels.json"))["categories"]])

spec = object_detector.SupportedModels.MOBILENET_MULTI_AVG
hparams = object_detector.HParams(export_dir="exported_model", epochs=EPOCHS, batch_size=8)
options = object_detector.ObjectDetectorOptions(supported_model=spec, hparams=hparams)
model = object_detector.ObjectDetector.create(train_data=train_data, validation_data=validation_data, options=options)

loss, coco_metrics = model.evaluate(validation_data, batch_size=4)
print("Loss:", loss)
print("COCO metrics:", coco_metrics)

model.export_model()
print(os.listdir("exported_model"))

## 4. Huấn luyện (lâu nhất, đừng đóng tab)
Xong sẽ in loss, chỉ số COCO và thư mục `exported_model`.

In [ ]:
!MPLBACKEND=Agg /content/venv/bin/python train.py

## 5. Tải mô hình về
Tải xong, chép `model.tflite` vào điện thoại rồi trong app bấm **📥 Nạp mô hình**.

In [ ]:
from google.colab import files
!ls -la exported_model
files.download('exported_model/model.tflite')